# 01 — WaveNet: A Generative Model for Raw Audio

**Paper:** *WaveNet: A Generative Model for Raw Audio* (van den Oord et al., DeepMind 2016)  
**arXiv:** https://arxiv.org/abs/1609.03499

---

## Why WaveNet?

Before WaveNet, neural TTS used **concatenative synthesis** (stitching recorded speech fragments)  
or **parametric synthesis** (predicting vocoder parameters like F0, spectral envelope).  
Both produce robotic, unnatural audio.

**WaveNet** generates audio **one sample at a time** directly in the waveform domain.  
At 16,000 Hz, that means 16,000 predictions per second — and the result sounds human.

Key idea: model the joint probability of a waveform as a product of conditional distributions:

$$p(x) = \prod_{t=1}^{T} p(x_t \mid x_1, \ldots, x_{t-1})$$

Each sample depends on **all previous samples** — this is an autoregressive model.

## Architecture: Dilated Causal Convolutions

WaveNet uses **causal convolutions** (no future leakage) stacked with **dilation** to grow the receptive field exponentially without increasing parameters.

![WaveNet Dilated Convolutions](./figures/wavenet_dilated.png)

**Dilation** doubles at each layer: 1, 2, 4, 8, 16, 32, ...  
After k layers: receptive field = 2^k samples

With 30 layers and dilation up to 512:  
Receptive field = **~6,000 samples** = 375 ms at 16 kHz — enough context for natural speech.

```
Input:  x_{t-R}, ..., x_{t-2}, x_{t-1}
                                    |
                        Causal Conv (d=1)
                        Causal Conv (d=2)
                        Causal Conv (d=4)
                              ...
                        Causal Conv (d=512)
                                    |
                              Residual + Skip
                                    |
                            Softmax over 256 values
                                    |
                        Predicted sample x_t
```

**mu-law quantization:** raw audio (float) is quantized to 256 levels (8-bit)  
so the output is a 256-class classification problem per sample.

In [ ]:
# !pip install torch numpy matplotlib

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## mu-Law Quantization

Raw audio values are continuous floats in [-1, 1].  
WaveNet turns this into a **256-class classification** using mu-law companding —  
the same compression used in telephone systems (ITU G.711).

$$\text{encode}(x) = \text{sign}(x) \cdot \frac{\ln(1 + \mu|x|)}{\ln(1 + \mu)}, \quad \mu = 255$$

This compresses loud sounds and expands quiet ones — matching human loudness perception.

In [ ]:
# ── mu-Law Encoding / Decoding ──

def mu_law_encode(x, mu=255):
    x = x.clamp(-1, 1)
    x_mu = x.sign() * torch.log1p(mu * x.abs()) / math.log(1 + mu)
    return ((x_mu + 1) / 2 * mu + 0.5).long()   # quantize to [0, 255]

def mu_law_decode(x_mu, mu=255):
    x = 2 * x_mu.float() / mu - 1               # back to [-1, 1]
    return x.sign() * (torch.exp(x.abs() * math.log(1 + mu)) - 1) / mu

# Visualize
x = torch.linspace(-1, 1, 1000)
encoded = mu_law_encode(x)
decoded = mu_law_decode(encoded)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x.numpy(), encoded.numpy(), color='steelblue')
axes[0].set_title('mu-Law Encoding: float -> 256 quantization levels')
axes[0].set_xlabel('Input amplitude'); axes[0].set_ylabel('Quantized level (0-255)')

axes[1].plot(x.numpy(), decoded.numpy(), color='tomato', label='decoded')
axes[1].plot(x.numpy(), x.numpy(), 'k--', alpha=0.4, label='original (ideal)')
axes[1].set_title('mu-Law Decode: reconstruction quality')
axes[1].set_xlabel('Original'); axes[1].set_ylabel('Decoded')
axes[1].legend()

plt.tight_layout(); plt.show()

# Round-trip error
print(f"Max reconstruction error: {(x - decoded).abs().max():.5f}")
print(f"Quantization bins: 256  ->  {2/256*1000:.1f} ms dynamic range")

In [ ]:
# ── Causal Convolution with Dilation ──

class CausalDilatedConv(nn.Module):
    def __init__(self, channels, kernel_size=2, dilation=1):
        super().__init__()
        self.dilation = dilation
        self.padding  = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(channels, channels * 2,   # *2 for gated activation
                              kernel_size=kernel_size,
                              dilation=dilation,
                              padding=self.padding)
        self.res_proj  = nn.Conv1d(channels, channels, 1)
        self.skip_proj = nn.Conv1d(channels, channels, 1)

    def forward(self, x):
        # Remove future context (causal)
        h = self.conv(x)[..., :-self.padding] if self.padding > 0 else self.conv(x)
        # Gated activation: tanh(gate1) * sigmoid(gate2)
        h_tanh, h_sig = h.chunk(2, dim=1)
        h = torch.tanh(h_tanh) * torch.sigmoid(h_sig)
        # Residual + skip
        return self.res_proj(h) + x, self.skip_proj(h)


class WaveNet(nn.Module):
    def __init__(self, n_channels=64, n_layers=10, n_stacks=3,
                 kernel_size=2, out_classes=256):
        super().__init__()
        self.input_conv = nn.Conv1d(1, n_channels, 1)
        self.layers = nn.ModuleList()
        for stack in range(n_stacks):
            for i in range(n_layers):
                dilation = 2 ** i
                self.layers.append(CausalDilatedConv(n_channels, kernel_size, dilation))

        self.output = nn.Sequential(
            nn.ReLU(),
            nn.Conv1d(n_channels, n_channels, 1), nn.ReLU(),
            nn.Conv1d(n_channels, out_classes, 1)
        )

        total_rf = sum(2**i for i in range(n_layers)) * n_stacks + 1
        print(f"WaveNet: {n_stacks} stacks x {n_layers} layers")
        print(f"Receptive field: ~{total_rf} samples = {total_rf/16000*1000:.0f} ms at 16kHz")
        total = sum(p.numel() for p in self.parameters()) / 1e6
        print(f"Parameters: {total:.1f}M")

    def forward(self, x):
        # x: (B, 1, T) raw waveform
        h = self.input_conv(x)
        skip_sum = 0
        for layer in self.layers:
            h, skip = layer(h)
            skip_sum = skip_sum + skip
        return self.output(skip_sum)   # (B, 256, T)


model = WaveNet(n_channels=32, n_layers=8, n_stacks=2).to(device)

# Test forward pass
x_test = torch.randn(2, 1, 256).to(device)
logits  = model(x_test)
print(f"\nInput:  {x_test.shape}")
print(f"Output: {logits.shape}  (B, 256 classes, T)")

In [ ]:
# ── Receptive Field Visualization ──

n_layers = 10
dilations = [2**i for i in range(n_layers)]

fig, ax = plt.subplots(figsize=(12, 4))
colors = plt.cm.Blues(np.linspace(0.3, 1.0, n_layers))

for i, d in enumerate(dilations):
    label = f"Layer {i+1}  (d={d})"
    ax.barh(i, d * 2, left=-d, height=0.7, color=colors[i], alpha=0.85, label=label)
    ax.text(d + 0.3, i, f"d={d}", va='center', fontsize=9)

ax.set_yticks(range(n_layers))
ax.set_yticklabels([f"Layer {i+1}" for i in range(n_layers)], fontsize=9)
ax.axvline(0, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Samples relative to current position (t=0)')
ax.set_title(f'WaveNet Dilation Pattern — Receptive Field Grows Exponentially\n'
             f'Total receptive field: {sum(dilations)*2} samples = {sum(dilations)*2/16000*1000:.0f}ms at 16kHz')
plt.tight_layout(); plt.show()

## Training WaveNet

WaveNet is trained as a **cross-entropy classification** problem:  
given the context x_1 ... x_{t-1}, predict the correct quantized value of x_t.

**Conditional WaveNet (for TTS):**  
Add a conditioning signal (e.g. mel spectrogram or linguistic features) via  
a 1x1 conv that adds to every layer — this is the "global conditioning" mechanism.

```
Input conditioning:  mel spectrogram h  (B, n_mels, T')
    |  Upsample to audio rate
    v  (B, n_mels, T)
    +-- added into each WaveNet layer as bias
```

**Limitations of WaveNet (original):**
- Autoregressive: generates **one sample at a time** — extremely slow (~90 minutes for 1 second!)
- Inference speed: 16,000 predictions/second needed → impractical for real-time
- Fixed by: WaveRNN, WaveGlow, HiFi-GAN (parallel vocoders)

## Summary

| Component | Details |
|-----------|---------|
| **Architecture** | Dilated causal convolutions, stacked in residual blocks |
| **Activation** | Gated: tanh(W_f * x) * sigmoid(W_g * x) |
| **Output** | Softmax over 256 mu-law quantized values |
| **Receptive field** | ~6,000 samples (375 ms) with 30 layers |
| **Conditioning** | Add mel spectrogram or speaker embedding at each layer |
| **Key weakness** | Sequential generation = too slow for real-time inference |

**References:**
- WaveNet paper: [arxiv 1609.03499](https://arxiv.org/abs/1609.03499)
- WaveRNN (faster): [arxiv 1802.08435](https://arxiv.org/abs/1802.08435)